# Retail Sales Analysis — Portfolio Notebook v1.0

**ArtoWare Indonesia**

A business-facing analytical narrative for the Retail Sales Analysis portfolio project.

**Version:** v1.0.0 portfolio notebook  
**Dataset:** Superstore retail sales dataset  
**Scope:** data quality → business metrics → business analysis → visual evidence → business implications

> Run from the repository root with the project dependencies installed.

## 2. Data Loading & Quality

### Explain

The repository establishes a canonical retail schema before analysis. This keeps the notebook aligned with the reusable application pipeline.

### Execute

Load the configured dataset, normalize it into the canonical schema, and inspect its dimensions.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import pandas as pd
from config import DATASET_FILE
from src.loader import load_dataset
from src.schema import normalize_dataset
from src.cleaning import DataCleaner
from src.business_metrics import BusinessMetrics
from src.insights import BusinessInsights
from src.visualization import Visualizer

CLEAN_DATA = Path('data/processed/superstore_clean.csv')
IMAGE_DIR = Path('images')

df = normalize_dataset(load_dataset(DATASET_FILE))
print('Raw dataset:', DATASET_FILE)
print('Rows: {:,}'.format(len(df)))
print('Columns: {:,}'.format(len(df.columns)))

### Show

The first rows provide a visual check of the normalized dataset before cleaning.

In [ ]:
display(df.head())

### Interpret

The dataset is now inside the same canonical schema used by the application. Downstream analysis can therefore use the reusable cleaning, metrics, insights, and visualization layers without source-specific naming logic.

### Explain — Data Quality

Before cleaning, inspect missing values and duplicate records to establish the raw-data quality baseline.

### Execute

In [ ]:
missing = df.isna().sum()
duplicate_count = int(df.duplicated().sum())
print('Duplicate rows:', duplicate_count)
print('Columns with missing values:')
display(missing[missing > 0].to_frame('Missing Values') if (missing > 0).any() else pd.DataFrame({'Status':['No missing values detected']}))

### Show

In [ ]:
quality_summary = pd.DataFrame({'Check':['Rows','Columns','Duplicate Rows','Missing Cells'],'Value':[len(df),len(df.columns),duplicate_count,int(df.isna().sum().sum())]})
display(quality_summary)

### Interpret

This confirms the starting quality state before the reusable cleaning rules are applied.

## 3. Data Preparation

### Explain

Cleaning is performed through the reusable DataCleaner rather than maintaining a separate notebook implementation.

### Execute

In [ ]:
cleaned_df = DataCleaner(df, CLEAN_DATA).run()
print('Cleaned rows: {:,}'.format(len(cleaned_df)))
print('Clean dataset:', CLEAN_DATA)

### Show

In [ ]:
preparation_summary = pd.DataFrame({'Metric':['Raw Rows','Cleaned Rows','Rows Removed','Columns'],'Value':[len(df),len(cleaned_df),len(df)-len(cleaned_df),len(cleaned_df.columns)]})
display(preparation_summary)
display(cleaned_df.head())

### Interpret

The prepared dataset is now the controlled input for all business analysis. The notebook demonstrates the reusable result instead of duplicating validation logic.

## 4. Business Performance

### Explain

The KPI layer establishes the overall commercial baseline. Sales alone is insufficient, so profit, margin, orders, customers, and products are included.

### Execute

In [ ]:
kpis = BusinessMetrics(cleaned_df).run()

### Show

In [ ]:
kpi_table = pd.DataFrame({'Metric':['Total Sales','Total Profit','Profit Margin','Total Orders','Total Customers','Total Products'],'Value':['${:,.2f}'.format(kpis['total_sales']),'${:,.2f}'.format(kpis['total_profit']),'{:.2f}%'.format(kpis['profit_margin']),'{:,}'.format(kpis['total_orders']),'{:,}'.format(kpis['total_customers']),'{:,}'.format(kpis['total_products'])]})
display(kpi_table)

### Interpret

The KPI table establishes the scale and profitability context for every later comparison. Dimension-level results should be read against this overall baseline.

## 5. Business Analysis

### Explain

BusinessInsights evaluates category, region, month, customer, product, contribution, and profitability dimensions. Each result is shown before its interpretation.

### Execute

In [ ]:
results = BusinessInsights(cleaned_df).run()
category = results['category']['summary'].sort_values('Sales', ascending=False)
region = results['region']['summary'].sort_values('Sales', ascending=False)
monthly = results['monthly']['summary'].sort_index()
customer = results['customer']['top_10_customers'].to_frame('Sales')
product = results['product']['products'].sort_values('Sales', ascending=False)
contribution_category = results['contribution']['category']
contribution_region = results['contribution']['region']
profitability = results['profitability']

### Show — Category

In [ ]:
display(category[['Sales','Profit','Profit Margin']])

### Interpret — Category

Category performance should be read across sales, profit, and margin. Sales leadership does not automatically imply the strongest margin.

### Show — Region

In [ ]:
display(region[['Sales','Profit','Profit Margin']])

### Interpret — Region

Regional differences show where sales and profitability are concentrated. They are descriptive differences, not causal explanations.

### Show — Customer Concentration

In [ ]:
display(customer)

### Interpret — Customer

The top-customer view makes sales concentration visible. It does not by itself establish customer profitability or retention risk.

### Show — Product Performance

In [ ]:
display(product.head(10)[['Sales','Profit']])

### Interpret — Product

Leading products show where sales are concentrated. Separate profitability evidence is needed because high sales do not guarantee high profit.

### Show — Monthly Performance

In [ ]:
display(monthly[['Sales','Profit']])

### Interpret — Monthly

Monthly variation identifies stronger and weaker periods for investigation, but the dataset alone does not establish the causes of those changes.

### Show — Contribution

In [ ]:
print('Category contribution')
display(contribution_category)
print('Regional contribution')
display(contribution_region)

### Interpret — Contribution

Contribution adds scale context by showing which dimensions account for a larger share of the business. It should be read together with margin.

### Show — Profitability Risk

In [ ]:
risk_summary = pd.DataFrame({'Indicator':['Lowest-margin category','Lowest-margin region','Loss-making products'],'Value':[profitability['lowest_margin_category'],profitability['lowest_margin_region'],profitability['loss_making_product_count']]})
display(risk_summary)

### Interpret — Profitability Risk

These indicators identify areas for deeper investigation. They do not explain causality; operational, pricing, discount, product, or customer evidence would be needed for that.

## 6. Visual Evidence

### Explain

Tables establish analytical evidence; visuals make the major patterns easier to inspect and communicate. Each visual below keeps the Explain → Execute → Show → Interpret rhythm.

### Sales by Category — Explain

A focused visual provides an accessible view of the analytical pattern.

### Execute

In [ ]:
visualizer = Visualizer(cleaned_df, output_dir=IMAGE_DIR)
visualizer.sales_by_category(cleaned_df)

### Show

In [ ]:
display(Image(filename=str(IMAGE_DIR / 'sales_by_category.png')))

### Interpret

The category chart makes relative commercial scale immediately visible. Read it together with category profit and margin.

### Sales by Region — Explain

A focused visual provides an accessible view of the analytical pattern.

### Execute

In [ ]:
visualizer = Visualizer(cleaned_df, output_dir=IMAGE_DIR)
visualizer.sales_by_region(cleaned_df)

### Show

In [ ]:
display(Image(filename=str(IMAGE_DIR / 'sales_by_region.png')))

### Interpret

The regional chart reinforces the geographic concentration pattern identified in the tables.

### Monthly Sales Trend — Explain

A focused visual provides an accessible view of the analytical pattern.

### Execute

In [ ]:
visualizer = Visualizer(cleaned_df, output_dir=IMAGE_DIR)
visualizer.monthly_sales_trend(cleaned_df)

### Show

In [ ]:
display(Image(filename=str(IMAGE_DIR / 'monthly_sales_trend.png')))

### Interpret

The time-series chart makes peaks, troughs, and longer-term movement easier to inspect. These patterns are signals for investigation, not causal explanations.

### Profit by Category — Explain

A focused visual provides an accessible view of the analytical pattern.

### Execute

In [ ]:
visualizer = Visualizer(cleaned_df, output_dir=IMAGE_DIR)
visualizer.profit_analysis(cleaned_df)

### Show

In [ ]:
display(Image(filename=str(IMAGE_DIR / 'profit_by_category.png')))

### Interpret

The profit chart complements sales and demonstrates why revenue and profit are distinct business measures.

### Profit Margin by Category — Explain

A focused visual provides an accessible view of the analytical pattern.

### Execute

In [ ]:
visualizer = Visualizer(cleaned_df, output_dir=IMAGE_DIR)
visualizer.profit_margin_by_category(cleaned_df)

### Show

In [ ]:
display(Image(filename=str(IMAGE_DIR / 'profit_margin_by_category.png')))

### Interpret

The margin chart adds relative profitability context that absolute sales or profit can hide.

## 7. Business Insights & Implications

### Explain

The evidence can now be synthesized across KPIs, tables, and visuals. The goal is to connect observations into a business-readable narrative rather than repeat every output.

### Interpret

- **Scale and profitability are different dimensions.** Sales leadership should be read together with profit contribution and margin.
- **Regional performance varies.** Sales, profit, and margin can tell different stories.
- **Concentration matters.** Top customers and products show where commercial performance is concentrated.
- **Time matters.** Monthly trends identify stronger and weaker periods for further investigation.
- **Profitability risks require targeted analysis.** Low-margin dimensions and loss-making products can be hidden inside strong aggregate sales.

These are descriptive observations from the dataset, not causal claims.

### Business implications

The analysis identifies areas a business user could investigate next: margin pressure, loss-making products, concentration among leading customers or products, and periods with unusual sales movement. Further operational data would be needed to explain causes or prescribe actions.

## 8. Interactive Dashboard

### Explain

The static visuals in this notebook provide portfolio evidence. The project also includes an interactive Plotly dashboard for deeper exploration without duplicating the dashboard implementation here.

### Show

Interactive dashboard: `output/interactive/interactive_dashboard.html`

The README documents how the dashboard is generated and how the full pipeline is tested.

## 9. Conclusion

The final project combines reusable data cleaning and validation, business KPIs, structured insights, static visualization, an interactive Plotly dashboard, automated regression tests, and controlled dataset schema mapping.

The notebook is the **portfolio storytelling layer**. The `src/` package remains the reusable engineering layer.

The key portfolio principle is that every major analytical step is visible as:

**Explain → Execute → Show → Interpret**

This makes the notebook reproducible, readable, and demonstrative of analytical reasoning rather than simply presenting final charts.